In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import os
import sys

module_path = os.path.abspath(os.path.join("../.."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [17]:
from machines.predictions import RF_Prediction, SVR_Prediction, XGB_Prediction
from machines.enums import SvrKernelEnum
from preprocesses.preprocess import Preprocess
from preprocesses.enums import TransformEnum, TypeFileEnum, StandardScaleEnum
from metrics.error_metric import ErrorMetric
from metrics.enums import MetricEnum
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [18]:
def get_machine_metric(machine, file_csv):
    preprocessing = Preprocess(file_name=file_csv, type_file=TypeFileEnum.CSV)
    preprocessing.read_file(
        transform=TransformEnum.MEAN,
        standard_scale=StandardScaleEnum.BASIC,
    )
    preprocessing.build_train_and_test()
    x_train, y_train = preprocessing.get_train()

    # machine = RF_Prediction(n_estimators=1000)
    machine.training(x_train=x_train, y_train=y_train)

    x_test, y_test = preprocessing.get_test()
    y_predicted = machine.prediction(x_test=x_test)

    metric = ErrorMetric(y_predicted=y_predicted, y_test=y_test, x_test=x_test)
    metric.calculate_metric_prediction()
    return (
        metric.get_metric(metric=MetricEnum.MEAN_ABSOLUTE_PERCENTAGE_ERROR),
        metric.get_metric(metric=MetricEnum.ROOT_MEAN_SQUARED_ERROR),
        metric.get_metric(metric=MetricEnum.R2_SCORE),
    )

In [19]:

class CountProgress:
    def __init__(self, file_base: str = "progress_store") -> None:
        self.progress_json = f"test_{file_base}_progress.csv"
        self.end_json = f"test_{file_base}_end.csv"
        
    def set_total(self, total: int) -> None:
        with open(self.end_json, 'w') as f:
            json.dump(total, f)
        with open(self.progress_json, 'w') as f:
            json.dump(0, f)

    def increase(self):
        try:
            with open(self.progress_json, 'r') as f:
                count = json.load(f) 
                count += 1
            with open(self.progress_json, 'w') as f:
                json.dump(count, f)    
        except FileNotFoundError:
            return 0

In [20]:
import itertools

files = [
    {
        "file": "../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_m.csv",
        "key": "Monthly",
    },
    {
        "file": "../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_w.csv",
        "key": "Weekly",
    },
    {
        "file": "../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_f.csv",
        "key": "Fifthly",
    },
    {
        "file": "../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_d.csv",
        "key": "Daily",
    },
        {
        "file": "../../../data/obregon_1516_1617/original/o_57_o_phenotypic.csv",
        "key": "None",
    },
]
mape = 1
best = []
metrics = []
machines = ["RF", "SVR_LINEAL", "XGB"]
for file in files:
    print(file)
    df = pd.read_csv(file["file"])

    columns_no_ndvi = []
    for col in df.columns:
        if "ndvi" not in col.lower():
            columns_no_ndvi.append(col)

    # print(len(columns_no_ndvi))
    new_csv_file = "test_no_ndvi.csv"
    df[columns_no_ndvi].to_csv(new_csv_file, index=False)
    # print(columns_no_ndvi)
    columns_to_iterate = [
        "TGW",
        "HI",
        "BM",
        "CT_UAV_1",
        "CT_UAV_2",
        "CT_UAV_3",
        "CT_UAV_4",
    ]
    column_y = columns_no_ndvi[-1]

    # print(columns_to_iterate, column_y)
    df = pd.read_csv(new_csv_file)
    columns_selection = [
        x for x in itertools.product([True, False], repeat=len(columns_to_iterate))
    ][:-1]
    progress = CountProgress()
    progress.set_total(total=len(columns_selection) * len(machines))
    for columns in columns_selection:
        selected = np.array(columns_to_iterate)[np.array(columns)].tolist()
        selected = selected + columns_no_ndvi
        file_csv = "test_selection_characteristic.csv"
        df[selected].to_csv(file_csv, index=False)
        for machine_alias in machines:
            if machine_alias == "RF":
                machine = RF_Prediction(n_estimators=1000)
            elif machine_alias == "SVR_LINEAL":
                machine = SVR_Prediction(kernel=SvrKernelEnum.LINEAR)
            elif machine_alias == "XGB":
                machine = XGB_Prediction()

            new_mape, r_s_m_e, r2 = get_machine_metric(
                machine=machine, file_csv=file_csv
            )
            metrics.append(
                [
                    "-",
                    "-".join(np.array(columns_to_iterate)[np.array(columns)].tolist()),
                    file["key"],
                    machine_alias,
                    new_mape,
                    r_s_m_e,
                    r2,
                ]
            )
            progress.increase()
            if new_mape < mape:
                mape = new_mape
                best = selected

print(mape, best)
data = pd.DataFrame(
data=metrics,
columns=[
    "all variable",
    "variable cultivo",
    "variable climatica",
    "ml",
    "mape",
    "R_S_M_E",
    "r2",
],
)
data.to_csv("test_result_all.csv", index=False)

{'file': '../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_m.csv', 'key': 'Monthly'}
{'file': '../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_w.csv', 'key': 'Weekly'}


/usr/local/lib/python3.12/site-packages/sklearn/impute/_base.py:565: UserWarning: Skipping features without any observed values: [324]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/impute/_base.py:565: UserWarning: Skipping features without any observed values: [324]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/impute/_base.py:565: UserWarning: Skipping features without any observed values: [324]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/impute/_base.py:565: UserWarning: Skipping features without any observed values: [323]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/impute/_base.py:565: UserWar

{'file': '../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_f.csv', 'key': 'Fifthly'}
{'file': '../../../data/obregon_1516_1617/original/o_57_o_phenotypic_weather_d.csv', 'key': 'Daily'}
{'file': '../../../data/obregon_1516_1617/original/o_57_o_phenotypic.csv', 'key': 'None'}
0.03935918427394692 ['TGW', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4', 'TGW', 'HI', 'BM', 'CT_UAV_1', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4', '20151213-20160111_TempOut_AVG', '20151213-20160111_HiTemp_AVG', '20151213-20160111_LowTemp_AVG', '20151213-20160111_OutHum_AVG', '20151213-20160111_DewPt_AVG', '20151213-20160111_WindSpeed_AVG', '20151213-20160111_WindRun_AVG', '20151213-20160111_HiSpeed_AVG', '20151213-20160111_WindChill_AVG', '20151213-20160111_HeatIndex_AVG', '20151213-20160111_THWIndex_AVG', '20151213-20160111_Bar_AVG', '20151213-20160111_Rain_AVG', '20151213-20160111_RainRate_AVG', '20151213-20160111_SolarRad_AVG', '20151213-20160111_SolarEnergy_AVG', '20151213-20160111_HiSolarRad_AVG', '20151213